In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import fitsio
from pycorr import TwoPointCorrelationFunction, TwoPointEstimator, project_to_multipoles, project_to_wp, utils, setup_logging
from scipy.optimize import curve_fit
from astropy.table import Table

from dataloc import *
from desiclusteringtools import *
import plotting as pp
from desiclusteringtools import *

# MAKE ALL PLOTS TEXT BIGGER
plt.rcParams.update({'font.size': 15})
# But legend a bit smaller
plt.rcParams.update({'legend.fontsize': 12})
# Set DPI up a bit
plt.rcParams.update({'figure.dpi': 150})

# This notebook is for DESI Project 712

### Maps

In [ ]:
rpath = '/global/cfs/cdirs/desi/users/ianw89/groupcatalogs/BGS_Y3/v0.6/BGS_BRIGHT_17_full.ran.fits'
tbl = Table(fitsio.read(rpath))
filepath = '/global/cfs/cdirs/desi/users/ianw89/groupcatalogs/BGS_Y3/v0.6/GROUP_CATALOG_BGS_Y3_v0.6.fits'
groupcat = Table.read(filepath)
print(len(tbl))

In [ ]:
# Downsample to 10%
np.random.seed(42)
sel = np.random.rand(len(tbl)) < 0.033
tbl_s = tbl[sel]
sel2 = np.random.rand(len(groupcat)) < 0.1
groupcat_s = groupcat[sel2]

In [ ]:
#make map
f=pp.make_map(tbl_s['RA'].value, tbl_s['DEC'].value)
f=pp.make_map(groupcat_s['RA'].value, groupcat_s['DEC'].value)

### Get Early Sersic Test Results

In [ ]:
# Define the base directory where your clustering results are stored.
clustering_base_dir = '/global/cfs/cdirs/desi/users/ianw89/newclustering/DA2/LSS/loa-v1/LSScats/v1.1/'
all_results = load_allcounts_from_disk(clustering_base_dir)

# You can now access the data and parameters like this:
if all_results:
    print("\nExample of first loaded result:")
    first_result = all_results[0]
    print("Parameters:", first_result['params'])
    print("Data object:", first_result['data'])



In [ ]:
# Now, call the function with the loaded results and a specific weight type
plot_wp_QSF_bins(all_results, 'pip_bitwise')

In [ ]:
# Call the new function to generate the comparison plots
plot_weight_comparison(all_results)

Imaging Systematics Check

In [ ]:
# Find all *_linclusimsysfit.png files in the clustering_base_dir and its subdirectories
import glob
linclusimsysfit_files = glob.glob(os.path.join(clustering_base_dir, '**', '*_linclusimsysfit.png'), recursive=True) 
print(f"Found {len(linclusimsysfit_files)} linclusimsysfit.png files:")
for f in linclusimsysfit_files:
    print(f" - {f}")

In [ ]:
# First linclusimsysfit_files investigation
for i in range(10):
    first_file = linclusimsysfit_files[i]
    plt.figure(figsize=(10, 10))
    img = plt.imread(first_file)
    plt.imshow(img)
    plt.axis('off') 
    plt.show()

### Get the mag / g-r clustering test clustering

In [ ]:
# Define the base directory where your clustering results are stored.
#clustering_base_dir = '/global/cfs/cdirs/desi/users/ianw89/clustering712/DA2/LSS/loa-v1/LSScats/v1.1/v0.1'
clustering_base_dir = '/global/cfs/cdirs/desi/users/ianw89/clustering712/DA2/LSS/loa-v1/LSScats/v1.1/v0.2'
all_results = load_allcounts_from_disk(clustering_base_dir)

if all_results:
    print("\nExample of first loaded result:")
    first_result = all_results[0]
    print("Parameters:", first_result['params'])
    print("Data object:", first_result['data'])

In [ ]:
# Find the dataset we will use as the reference
for result in all_results:
    mag_range = result['params'].get('mag_range')
    gmr_range = result['params'].get('gr_range')  
    njack = result['params'].get('njack')
    if mag_range is None and gmr_range is None and int(njack) == 64:
        reference_result = result

ref_estimator = reference_result['data']
rp_ref, wp_ref, cov_ref = ref_estimator.get_corr(return_sep=True, return_cov=True, mode='wp')

In [ ]:
# Filter to chosen weighting scheme
filtered_results = [res for res in all_results if res['params']['weights'] == 'pip_bitwise']

# Filter to chosen number of jackknife regions
filtered_results = [res for res in filtered_results if int(res['params'].get('njack')) == 64]

print(f"Kept {len(filtered_results)} results.")

outdir = '/global/cfs/cdirs/desi/users/ianw89/clustering712/DA2/LSS/loa-v1/LSScats/v1.1/export/'
#save_wp_for_maggmr(outdir, filtered_results)

In [ ]:
plot_wp_mag_gmr(filtered_results, 'pip_bitwise')

In [ ]:
# Group results by magnitude range
results_by_mag = {}
for result in filtered_results:
    mag_range = result['params'].get('mag_range')
    if mag_range is None:
        continue
    if mag_range not in results_by_mag:
        results_by_mag[mag_range] = []
    results_by_mag[mag_range].append(result)

# Order by magnitude range 
results_by_mag = dict(sorted(results_by_mag.items(), key=lambda x: float(x[0].split('to')[0])))

#assert len(results_by_mag) == 10

In [ ]:
# Fit to powerlaw and find offset method
#
#for mag_range, results in results_by_mag.items():
#    print(f"Processing magnitude range: {mag_range}")
#    for result in results:
#        rp, wp = result['data'].get_corr(return_sep=True, return_cov=False, mode='wp')
#        info = compare_wp_powerlaws((rp_ref, wp_ref, cov_ref), (rp, wp))
#        print(info)
#        result['offset'] = info['offset']

In [ ]:
# Better. Calculate bias b compared to the reference.

with np.printoptions(precision=3, suppress=True):
    for mag_range, measurements in results_by_mag.items():
        #print("Mag range:", mag_range)
        for measurement in measurements:
            rp, wp, cov = measurement['data'].get_corr(return_sep=True, return_cov=True, mode='wp')
            bias_info = get_bias((rp_ref, wp_ref, cov_ref), (rp, wp, cov))
            measurement['bias'] = bias_info[0]
            measurement['bias_err_up'] = bias_info[1]
            measurement['bias_err_down'] = bias_info[2]
            #print("  g-r: ", measurement['params'].get('gr_range'), "Bias info:", np.array(bias_info))

In [ ]:
# 2 rows of 5 plots. One for each mag group. Plot the bias values for each color 
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
i = 0
for mag_range, measurements in results_by_mag.items():
    ax = axes[i]
    i += 1

    ax.errorbar(
        [np.mean([float(x) for x in m['params'].get('gr_range', '0to0').split('to')]) for m in measurements],
        [m['bias'] for m in measurements],
        yerr=[[m['bias_err_down'] for m in measurements], [m['bias_err_up'] for m in measurements]],
        fmt='.',
        capsize=3,
    )

    ax.set_xlim([0.25, 1.25])
    ax.set_ylim([0.5, 2.5])  
    ax.set_xlabel('g-r')
    ax.set_ylabel('b')
    ax.set_title(f'{mag_range}')
    
plt.tight_layout()

In [ ]:
# Save the filtered results to a file
# Just need the mag range, g-r range, and bias info

# Also need to save full reference data. rp, wp, cov

In [ ]:
"""
Mag range: -22.6473to-21.9413
  g-r:  0.9213to0.9778 Bias info: [1.991 0.027 0.026]
Mag range: -21.9413to-21.4880
  g-r:  0.9696to1.0762 Bias info: [1.703 0.023 0.022]
  g-r:  0.4654to0.6689 Bias info: [1.167 0.03  0.03 ]
Mag range: -21.0910to-20.7092
  g-r:  0.5561to0.6948 Bias info: [1.061 0.016 0.016]
Mag range: -20.7092to-20.3206
  g-r:  0.5105to0.6492 Bias info: [0.997 0.016 0.016]
Mag range: -20.3206to-19.9002
  g-r:  0.2675to0.4793 Bias info: [0.922 0.022 0.022]
  g-r:  0.9016to1.0363 Bias info: [1.053 0.015 0.015]
  g-r:  0.6136to0.7562 Bias info: [0.972 0.015 0.015]
Mag range: -19.9002to-19.4147
  g-r:  0.8768to1.0244 Bias info: [0.95  0.018 0.018]
Mag range: -19.4147to-18.8037
  g-r:  0.4279to0.5416 Bias info: [0.894 0.018 0.018]
  g-r:  0.8428to1.0114 Bias info: [0.981 0.018 0.018]
Mag range: -18.8037to-17.9408
  g-r:  0.3963to0.4970 Bias info: [0.818 0.021 0.021]
  g-r:  0.6074to0.7760 Bias info: [1.    0.028 0.028]
"""

In [ ]:
# TODO Worry. 
# I am getting different bias values (not just errors) when I either don't use target cov matrix vs use cov with 4 jackknife regions vs 64 jackknife regions.